# GAP 7 — Pooling Ablation on the N-CMAPSS Sequential CNN-LSTM

**Purpose (from the reviewer's GAP 7).** Rerun the Bayesian search on the sequential model with
**pooling removed**, using the same trial budget, and report whether the best configuration now
**retains LSTM layers**.

**Why it matters.** With pooling, the optimizer consistently selected a CNN-only design
(0 LSTM layers). The paper attributes this to pooling compressing the temporal resolution before
the recurrent layers. If, with pooling removed, the search **keeps** the LSTM, that bottleneck
claim moves from *inferred* to *demonstrated* — the single most valuable remaining experiment.

This notebook does **only** the ablation; the baseline results already have evidence in the repo
and are not recomputed here.

- Data: `N-CMAPSS_DS05.h5` (expects to run from the `N-Cmapps` folder that holds the file).
- Same search space as the original sequential model **except no MaxPooling**.
- 30 Bayesian trials, seed 42, isolated test units 2, 4, 8. Run on a CUDA GPU (RTX 3090).

In [ ]:
import os, json
import numpy as np, pandas as pd, h5py
from pandas import DataFrame
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout, Flatten, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt

SEED = 42
os.environ['PYTHONHASHSEED']=str(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
# ---- config ----
# This notebook expects to sit in / be run from the N-Cmapps folder that contains the .h5.
# If you run it from elsewhere, set DATA_PATH to the full path of  ...\N-Cmapps\N-CMAPSS_DS05.h5
DATA_PATH = 'N-CMAPSS_DS05.h5'
POOLING   = 'none'     # 'none' = pooling removed (the GAP 7 ablation) ; 'single' = one pooling layer only
MAX_TRIALS = 30

In [ ]:
with h5py.File(DATA_PATH, 'r') as hdf:
    W_dev=np.array(hdf.get('W_dev'));   X_s_dev=np.array(hdf.get('X_s_dev')); A_dev=np.array(hdf.get('A_dev')); Y_dev=np.array(hdf.get('Y_dev'))
    W_test=np.array(hdf.get('W_test')); X_s_test=np.array(hdf.get('X_s_test')); A_test=np.array(hdf.get('A_test')); Y_test=np.array(hdf.get('Y_test'))
    W_var=list(np.array(hdf.get('W_var'),dtype='U20')); X_s_var=list(np.array(hdf.get('X_s_var'),dtype='U20')); A_var=list(np.array(hdf.get('A_var'),dtype='U20'))

W=np.concatenate((W_dev,W_test),0); X_s=np.concatenate((X_s_dev,X_s_test),0)
A=np.concatenate((A_dev,A_test),0); Y=np.concatenate((Y_dev,Y_test),0)

X = DataFrame(np.concatenate((W, X_s, A, Y), 1), columns=W_var+X_s_var+A_var+['Y'])
Yd = DataFrame(np.concatenate((A, Y), 1), columns=A_var+['Y'])
for col in ['cycle','Fc','hs']:
    Yd.drop(col, axis=1, inplace=True); X.drop(col, axis=1, inplace=True)
X.drop('alt', axis=1, inplace=True)   # 3 operational settings + 14 sensors = 17 features

TRAIN_UNITS=[1,3,5,6,7,9,10]; TEST_UNITS=[2,4,8]
Xtr = X[X.unit.isin(TRAIN_UNITS)].copy(); Ytr = Yd[Yd.unit.isin(TRAIN_UNITS)].copy()
Xte = X[X.unit.isin(TEST_UNITS)].copy();  Yte = Yd[Yd.unit.isin(TEST_UNITS)].copy()
for d in (Xtr,Xte): d.drop(['unit','Y'], axis=1, inplace=True)
for d in (Ytr,Yte): d.drop('unit', axis=1, inplace=True)

sc = MinMaxScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
Xtr = Xtr.reshape(Xtr.shape[0], Xtr.shape[1], 1); Xte = Xte.reshape(Xte.shape[0], Xte.shape[1], 1)
Ytr = Ytr.values; Yte = Yte.values; N_FEAT = Xtr.shape[1]

# clean validation split from TRAINING units only (test stays isolated)
Xtr_s, Xval, Ytr_s, Yval = train_test_split(Xtr, Ytr, test_size=0.2, random_state=SEED)
print('train', Xtr.shape, '| search-train', Xtr_s.shape, '| val', Xval.shape, '| isolated test', Xte.shape)

## Ablated sequential model — **no pooling**

Identical search space to the original sequential model (CNN 1–3, filters 32–128, kernel {3,5,7},
LSTM **0–2**, dense, Adam), but `MaxPooling1D` is removed so the full-resolution sequence reaches
the recurrent layers. The `num_lstm` hyperparameter is what we watch.

In [ ]:
def build_ablation(hp):
    inp = Input(shape=(N_FEAT, 1)); x = inp
    n_conv = hp.Int('num_conv', 1, 3)
    for i in range(n_conv):
        x = Conv1D(hp.Int(f'conv_{i}_filters', 32, 128, step=16),
                   hp.Choice(f'conv_{i}_kernel', [3,5,7]), padding='same', activation='relu')(x)
        # --- ABLATION: pooling removed (POOLING='none'); optional single pooling after first conv ---
        if POOLING == 'single' and i == 0:
            x = MaxPooling1D(2, padding='same')(x)
    n_lstm = hp.Int('num_lstm', 0, 2)
    for i in range(n_lstm):
        x = LSTM(hp.Int(f'lstm_{i}_units', 32, 128, step=16), return_sequences=(i < n_lstm-1))(x)
    if n_lstm == 0:
        x = Flatten()(x)
    x = Dense(hp.Int('dense_units', 32, 128, step=16), activation='relu')(x)
    x = Dropout(hp.Float('dropout', 0.0, 0.5, step=0.1))(x)
    out = Dense(1, activation='linear')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(hp.Choice('lr', [1e-2,1e-3,1e-4])),
              loss='mean_squared_error', metrics=['mean_absolute_error'])
    return m

tuner = kt.BayesianOptimization(build_ablation, objective='val_loss', max_trials=MAX_TRIALS,
        seed=SEED, directory='gap7_ablation', project_name=f'seq_pool_{POOLING}', overwrite=True)
tuner.search(Xtr_s, Ytr_s, validation_data=(Xval, Yval), epochs=20, batch_size=32,
             callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
                        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)], verbose=1)
best = tuner.get_best_hyperparameters(1)[0]
print('\nBest ablated-sequential hyperparameters:')
for k,v in best.values.items(): print(f'  {k}: {v}')

In [ ]:
# retrain best on full training pool (10% held out for early stopping) and evaluate on isolated test
model = tuner.hypermodel.build(best)
model.fit(Xtr, Ytr, epochs=50, batch_size=32, validation_split=0.1, verbose=1,
          callbacks=[EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
                     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)])
p = model.predict(Xte, batch_size=256, verbose=0).flatten(); y = Yte.flatten()
test_rmse = float(np.sqrt(mean_squared_error(y,p)))
print(f'\nAblation TEST  RMSE={test_rmse:.4f}  MAE={mean_absolute_error(y,p):.4f}  R2={r2_score(y,p):.4f}')

In [ ]:
n_lstm = int(best.get('num_lstm'))
print('='*60)
print('GAP 7 ABLATION RESULT  (pooling =', POOLING + ')')
print('='*60)
print(f'  LSTM layers selected with pooling removed : {n_lstm}')
print(f'  Original sequential (with pooling)        : 0  (CNN-only)')
print(f'  Ablation test RMSE                        : {test_rmse:.4f}')
print('-'*60)
if n_lstm > 0:
    print('LSTM RETAINED once pooling is removed  ->  the pooling-induced temporal')
    print('bottleneck is DEMONSTRATED, not merely inferred.')
else:
    print('LSTM still not selected  ->  the bottleneck is not attributable to pooling alone;')
    print('report this honestly and keep the mechanism stated as inferred.')

json.dump({'pooling':POOLING, 'num_lstm_selected':n_lstm, 'test_rmse':round(test_rmse,4),
           'original_with_pooling_num_lstm':0, 'lstm_retained':bool(n_lstm>0)},
          open('gap7_ablation_result.json','w'), indent=2)
print('\nsaved -> gap7_ablation_result.json')